# NJ Benchmark Interactive Analysis

This notebook provides interactive exploration of NJ benchmark results.

## Setup

In [1]:
import sys
import os

# Add parent directory to path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from nj_utils import (
    quick_summary, 
    compare_methods, 
    find_best_method,
    method_consistency,
    seed_analysis
)

# Notebook display settings
%matplotlib inline
plt.style.use('seaborn-v0_8-paper')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

## Load Data

In [2]:
# Load benchmark results
df_benchmark = pd.read_csv('results/nj_benchmark_results.csv')
df_benchmark = df_benchmark[df_benchmark['status'] == 'success']

# Load summary statistics
df_summary = pd.read_csv('results/nj_summary_statistics.csv')

# Load statistical tests
df_tests = pd.read_csv('results/nj_statistical_tests.csv')

print(f"Loaded {len(df_benchmark)} successful benchmark runs")
print(f"Methods: {df_benchmark['method'].nunique()}")
print(f"Seeds: {df_benchmark['seed'].nunique()}")

FileNotFoundError: [Errno 2] No such file or directory: 'results/nj_benchmark_results.csv'

## Quick Summary

In [ ]:
quick_summary()

## Top Methods Analysis

In [ ]:
# Find best methods for multiset F1
best_multiset = find_best_method('multiset_f1')

In [ ]:
# Find best methods for unique F1
best_unique = find_best_method('unique_f1')

## Method Consistency

In [ ]:
consistency = method_consistency(top_n=15)

## Visualizations

In [ ]:
# Distribution of F1 scores for top 10 methods
top_methods = df_benchmark.groupby('method')['multiset_f1'].mean().nlargest(10).index
df_top = df_benchmark[df_benchmark['method'].isin(top_methods)]

plt.figure(figsize=(14, 6))
sns.boxplot(data=df_top, x='method', y='multiset_f1', palette='Set2')
plt.xticks(rotation=45, ha='right')
plt.title('Multiset F1 Score Distribution (Top 10 Methods)', fontsize=14, fontweight='bold')
plt.ylabel('Multiset F1 Score')
plt.xlabel('Method')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Multiset F1 vs Unique F1
plt.figure(figsize=(10, 8))

for method in df_benchmark['method'].unique():
    method_df = df_benchmark[df_benchmark['method'] == method]
    if method == 'ctbs':
        plt.scatter(method_df['multiset_f1'], method_df['unique_f1'], 
                   label=method, s=100, alpha=0.8, edgecolors='black', linewidth=2, zorder=10)
    else:
        plt.scatter(method_df['multiset_f1'], method_df['unique_f1'], 
                   label=method, s=50, alpha=0.6)

plt.xlabel('Multiset F1 Score', fontsize=12)
plt.ylabel('Unique F1 Score', fontsize=12)
plt.title('Performance Comparison: Multiset vs Unique F1', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Statistical Significance

In [ ]:
# Show significant results
sig_results = df_tests[df_tests['significant_005_bonf'] == True]
print(f"\nSignificant results (p < 0.05, Bonferroni corrected): {len(sig_results)}\n")

if len(sig_results) > 0:
    display(sig_results[['method', 'metric', 'pvalue', 'pvalue_corrected', 'cohens_d', 
                         'mean_baseline', 'mean_method', 'wins', 'losses']])
else:
    print("No significant differences found after Bonferroni correction.")

## Head-to-Head Comparisons

In [ ]:
# Compare CTBS vs best NJ variant
best_nj = df_benchmark[df_benchmark['method'] != 'ctbs'].groupby('method')['multiset_f1'].mean().idxmax()
print(f"Comparing CTBS vs {best_nj}\n")
compare_methods('ctbs', best_nj)

## Seed-Level Analysis

In [ ]:
# Analyze a specific seed
sample_seed = df_benchmark['seed'].iloc[0]
print(f"Analyzing seed: {sample_seed}\n")
seed_analysis(sample_seed)

## Custom Analysis

In [ ]:
# Example: Count how many times each method beats CTBS
ctbs_df = df_benchmark[df_benchmark['method'] == 'ctbs'].set_index('seed')

wins_against_ctbs = {}

for method in df_benchmark['method'].unique():
    if method == 'ctbs':
        continue
    
    method_df = df_benchmark[df_benchmark['method'] == method].set_index('seed')
    common_seeds = ctbs_df.index.intersection(method_df.index)
    
    if len(common_seeds) > 0:
        wins = np.sum(method_df.loc[common_seeds, 'multiset_f1'] > 
                     ctbs_df.loc[common_seeds, 'multiset_f1'])
        wins_against_ctbs[method] = (wins, len(common_seeds))

print("\nMethods that beat CTBS (Multiset F1):")
print("-" * 60)
for method, (wins, total) in sorted(wins_against_ctbs.items(), key=lambda x: x[1][0], reverse=True):
    pct = 100 * wins / total
    print(f"{method:35s} | {wins:3d}/{total:3d} ({pct:5.1f}%)")

## Export Results

In [ ]:
# Export top 10 methods table
from nj_utils import export_top_methods_table

top_table = export_top_methods_table(n=10, output_file='results/top_10_methods.csv')
display(top_table)